In [1]:
import sys
import warnings
from config import Config
from pipeline import Pipeline

warnings.filterwarnings('ignore')

config = Config()

config.csv_path = '../paper_verses.csv'
config.output_dir = './results'
config.use_scratch = False

config.subset_fraction = 0.2
config.poem_subset_fraction = 0.3

config.batch_size = 32

config.verse_similarity_thresholds = [0.5,0.6]
config.leiden_resolutions = [0.4, 0.5]
config.n_bootstraps = 3
config.n_gridsearch_iterations = 3

config.poem_similarity_thresholds = [0.3, 0.4,0.5]

pipeline = Pipeline(config)
results = pipeline.run()

embedding: 100%|██████████| 44/44 [01:06<00:00,  1.51s/it]
WARNING clustering 1388 points to 100 centroids: please provide at least 3900 training points



embedding complete:
  time: 66.6s
  throughput: 20.9 texts/sec

pipeline completed
  performance report: ./performance_report.txt
  enriched dataset: ./enriched_dataset.csv

computational resources

cpu:
  cores: 14
  usage: 2.9%

system memory:
  total: 30.8 gb
  used: 20.0 gb (65.0%)
  available: 10.8 gb

gpus: none available


In [9]:
import pandas as pd
import numpy as np
df=pd.read_csv('enriched_dataset.csv')
def show_example_verse_clusters(df: pd.DataFrame, n_clusters: int = 3):
    clusters = (
        df['predicted_verse_cluster']
        .dropna()
        .astype(int)
        .unique()
    )
    cluster_sizes = (
    df.groupby("predicted_verse_cluster")
      .size()
      .rename("count")
    )

    clusters_with_multiple = cluster_sizes[cluster_sizes > 1].index.tolist()
    num_available = len(clusters_with_multiple)

    example_clusters = np.random.choice(
        clusters_with_multiple,
        size=min(n_clusters, num_available),
        replace=False
    )


    for cid in example_clusters:
        print(f"Cluster {cid}")

        subset = df[df['predicted_verse_cluster'] == cid]

        for _, row in subset.iterrows():
            print(f"Verse: {row['verse']}")

        print("\n")
show_example_verse_clusters(df, n_clusters=10)

Cluster 71
Verse: Τὸ πόνημα τὸ σύγγραμμα ἠ εὑτελὴς διὄπτρα
Verse: τὸ πόνημα τὸ σύγγραμμα, ἡ εὐτελὴς διόπτρα,
Verse: τὸ πόνημα, τὸ σύγγραμα, ἡ εὐτελὴς Διόπτρα,


Cluster 76
Verse: και ὑ θαλατεβοτε ε<ὑ>ρὴν | λιμυ μεν
Verse: οἱ θαλατεβοντες εὑρειν λιμ|ον,


Cluster 39
Verse: οὕτω καὶ οἰ ἀ(...)φωντες |
Verse: χ(ριστ)ε μου φοι|λατ(ε) τοῦς (…) τεικουντ(ας)
Verse: μι δαρῖς | κ(αὶ) ληπιθεὶς 
Verse: (...)κχεῖ (...) (...)μα· τῶν ἀειρρύτ(ων) λόγων: |
Verse: Ιδοὺ (καὶ) τοῦτο γράφο σοι (καὶ) ψιφήσας εὐρίσ(ης)·
Verse: ἀκακί(ας) δὲ κ(αὶ) πονηρί(ας) μεσον,
Verse: ፠ λόγων ὁ μάρκο(ς) τ(ὴν) ἀείρρυτον χύσ(ιν):·
Verse: ψυχ(ὴν) κατάρδων κ(αὶ) ποτίζ(ων) | τὰς φραίν(ας):-


Cluster 43
Verse: εἰδεῖν βιβλὴοὖ τέλος
Verse: οὕτως καὶ οἱ... βιβλίου τέλος... \\\\\\\\\ ἀμὴν +
Verse: εζτὐ [= ἔτσι]καὶ υ βιβλό[γρά]φοντες υδὴν βιβλήου τέλος
Verse: οντος και [..........] ηδὶν βιβλιου τελος.
Verse: οντος και [..........] ηδὶν βιβλιου τελος.
Verse: (...) βιβλίου τέλος
Verse: οὕτω | καὶ οἱ  βιβλογράφοντες εὐρεῖν βιβλίου τέλο

In [11]:
import pandas as pd

poems_df = pd.read_csv("enriched_dataset.csv")
poems_df["idoriginal_poem"] = poems_df["idoriginal_poem"].astype(str)

cluster_sizes = poems_df.groupby("predicted_poem_cluster").size()
multi_poem_clusters = cluster_sizes[cluster_sizes > 1].index.tolist()
print(f"Found {len(multi_poem_clusters)} multi-poem clusters.")

clusters_to_show = multi_poem_clusters[:3]
poems_to_show = poems_df[poems_df["predicted_poem_cluster"].isin(clusters_to_show)]

cluster_to_poems = (
    poems_to_show.groupby("predicted_poem_cluster")["idoriginal_poem"]
    .apply(list)
    .to_dict()
)

verse_dict = {}  
chunksize = 100_000

for chunk in pd.read_csv("../paper_verses.csv", chunksize=chunksize):
    chunk["idoriginal_poem"] = chunk["idoriginal_poem"].astype(str)
    chunk = chunk[chunk["idoriginal_poem"].isin(poems_to_show["idoriginal_poem"])]
    
    for poem_id, group in chunk.groupby("idoriginal_poem"):
        verses = group.sort_values("order")["verse"].tolist()
        verses = [str(v) for v in verses if pd.notna(v)]
        if poem_id in verse_dict:
            verse_dict[poem_id].extend(verses)
        else:
            verse_dict[poem_id] = verses

for cl in clusters_to_show:
    print("\n" + "="*80)
    print(f"CLUSTER {cl}")
    print("="*80)
    
    for poem_id in cluster_to_poems[cl]:
        poem_text = "\n".join(verse_dict.get(poem_id, []))
        print(f"\n--- Poem {poem_id} ---\n")
        print(poem_text)
        print("\n" + "-"*40)


Found 53 multi-poem clusters.

CLUSTER 0

--- Poem 16914 ---

+ ὥσπερ ξένοι χαίρουσιν ἰδεῖν | π(ατ)ρίδα·
οὕτω (καὶ) οἱ γρ(άφ)οντες [ἰδεῖν] βιβλίου τέλο(ς) +

----------------------------------------

--- Poem 16907 ---

+ ὥσπερ ξένοι χαίρο(...) πατρίδα,
οὕτω (καὶ) οἱ γράφοντες τέλος βιβλίου +

----------------------------------------

--- Poem 16982 ---

+ ὥσπερ ξένοι χαίροντες εἰδεῖν π(ατ)ρίδα |
(καὶ) οἱ θαλατεύοντες εὑρεῖν λιμέ|να·

----------------------------------------

--- Poem 16982 ---

+ ὥσπερ ξένοι χαίροντες εἰδεῖν π(ατ)ρίδα |
(καὶ) οἱ θαλατεύοντες εὑρεῖν λιμέ|να·

----------------------------------------

--- Poem 17003 ---

ὥσπερ ξένοι χαί|ροντὲς ἰδὴν | πατρίδ(α):
οὗτω | κ(αὶ) τοῖς γράφου|σιν βιβλήου | τέλος:-

----------------------------------------

--- Poem 17003 ---

ὥσπερ ξένοι χαί|ροντὲς ἰδὴν | πατρίδ(α):
οὗτω | κ(αὶ) τοῖς γράφου|σιν βιβλήου | τέλος:-

----------------------------------------

--- Poem 18496 ---

Ὥσπερ ξένοι χαίρουσιν ἰδεῖν π(ατ)ρίδα,
καὶ θαλαττεύον